# Методы оптимизации в машинном обучении
## Семинар 3. Квадратичная оптимизация, обусловленность и градиентный спуск — вычислительная часть

Эта часть семинара соответствует бумажным задачам 1–10. Математические идеи те же, но размерность или объём вычислений делают ручное решение неудобным.

**Рекомендуемое время:** около 40 минут. Если времени мало, в первую очередь сделайте задачи 1, 3, 5, 8, 9 и 10.

Во всех задачах используется евклидова норма `np.linalg.norm`, а визуализации строятся только через `matplotlib`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)


---
## Задача 1. Квадратичная функция в высокой размерности

Сгенерируем симметричную матрицу $A\in\mathbb R^{20\times20}$, вектор $b\in\mathbb R^{20}$ и функцию

$$
f(x)=\frac12x^\top Ax-b^\top x.
$$

1. Реализуйте `f(x)` и `grad_f(x)` по формулам
   $$
   f(x)=\frac12x^\top Ax-b^\top x,
   \qquad
   \nabla f(x)=Ax-b.
   $$
2. В случайной точке $x$ проверьте градиент методом центральных конечных разностей:
   $$
   \frac{\partial f}{\partial x_i}(x)
   \approx
   \frac{f(x+\varepsilon e_i)-f(x-\varepsilon e_i)}{2\varepsilon}.
   $$
3. Выведите норму ошибки между аналитическим и численным градиентом.
4. Повторите проверку для нескольких значений $\varepsilon$ от $10^{-2}$ до $10^{-8}$ и постройте график ошибки.
5. Объясните, почему при слишком маленьком $\varepsilon$ ошибка перестаёт уменьшаться.


In [ ]:
d = 20
M = rng.normal(size=(d, d))
A = 0.5 * (M + M.T)
b = rng.normal(size=d)
x = rng.normal(size=d)


def f(x):
    # TODO
    pass


def grad_f(x):
    # TODO
    pass


def finite_difference_grad(f, x, eps):
    g = np.zeros_like(x, dtype=float)
    # TODO: центральные конечные разности по каждой координате
    return g

# TODO: сравните grad_f(x) и finite_difference_grad(...)

# TODO: ошибки для eps = 10^{-2},...,10^{-8}
# TODO: построить отдельный график error(eps) в log-log масштабе


---
## Задача 2. Спектр и положительная определённость

Постройте три симметричные матрицы размера $30\times30$ с заранее заданными спектрами:

- $A_{+}$: все собственные значения положительны;
- $A_{0}$: одно собственное значение равно нулю, остальные положительны;
- $A_{\pm}$: есть и положительные, и отрицательные собственные значения.

Для каждой матрицы:

1. вычислите собственные значения через `np.linalg.eigvalsh`;
2. классифицируйте матрицу как положительно определённую, положительно полуопределённую или неопределённую;
3. сгенерируйте 20 000 случайных единичных векторов $x$;
4. вычислите $x^\top Ax$ для каждого вектора;
5. сравните минимальные и максимальные наблюдаемые значения с $\lambda_{\min}$ и $\lambda_{\max}$;
6. объясните, почему случайное тестирование квадратичной формы не заменяет спектральный критерий.


In [ ]:
def random_orthogonal(d, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
    return Q


def matrix_with_spectrum(eigvals, rng):
    Q = random_orthogonal(len(eigvals), rng)
    return Q @ np.diag(eigvals) @ Q.T


d = 30
A_pos = matrix_with_spectrum(np.linspace(0.5, 5.0, d), rng)
A_psd = matrix_with_spectrum(np.r_[0.0, np.linspace(0.5, 5.0, d-1)], rng)
A_indef = matrix_with_spectrum(np.linspace(-3.0, 5.0, d), rng)

matrices = {
    "positive definite": A_pos,
    "positive semidefinite": A_psd,
    "indefinite": A_indef,
}

# TODO: для каждой матрицы eigvals = np.linalg.eigvalsh(A)
# TODO: классификация по знакам eigvals
# TODO: сгенерировать X размера (20000, d), нормировать строки
# TODO: quadratic = np.einsum('bi,ij,bj->b', X, A, X)
# TODO: сравнить min/max quadratic с крайними eigvals


---
## Задача 3. Точный минимум квадратичной функции

Сгенерируем положительно определённую матрицу $A\in\mathbb R^{100\times100}$ и случайный вектор $b$. Рассмотрим

$$
f(x)=\frac12x^\top Ax-b^\top x.
$$

1. Найдите точный минимум
   $$
   x^\star=A^{-1}b
   $$
   через `np.linalg.solve(A, b)`.
2. Проверьте численно $\|Ax^\star-b\|_2\approx0$.
3. Для 1000 случайных точек $x$ проверьте тождество
   $$
   f(x)-f(x^\star)
   =
   \frac12(x-x^\star)^\top A(x-x^\star).
   $$
4. Проверьте, что правая часть неотрицательна.
5. Сравните `np.linalg.solve(A, b)` и явное вычисление `np.linalg.inv(A) @ b` по времени и по невязке. Какой способ предпочтительнее?


In [ ]:
d = 100
Q = random_orthogonal(d, rng)
eigvals = np.geomspace(1.0, 50.0, d)
A = Q @ np.diag(eigvals) @ Q.T
b = rng.normal(size=d)


def quad_value(x):
    return 0.5 * x @ A @ x - b @ x

# TODO: x_star = np.linalg.solve(A, b)
# TODO: residual = ||A @ x_star - b||

# TODO: 1000 случайных x и проверка identity_error

# Необязательно: сравните время solve и inv через %timeit в Colab


---
## Задача 4. Собственные направления и линии уровня

Рассмотрим семейство матриц

$$
A(\theta)=Q(\theta)
\begin{pmatrix}1&0\\0&16\end{pmatrix}
Q(\theta)^\top,
$$

где $Q(\theta)$ — матрица поворота.

1. Для $\theta\in\{0,\pi/6,\pi/3\}$ постройте линии уровня
   $$
   f(x)=\frac12x^\top A(\theta)x.
   $$
2. На каждый график нанесите собственные векторы матрицы $A(\theta)$.
3. Подпишите соответствующие собственные значения.
4. Проверьте численно, что собственные значения не зависят от $\theta$.
5. Сформулируйте вывод: что меняет ортогональный поворот и что он не меняет?


In [ ]:
def rotation(theta):
    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)],
    ])


def contour_for_A(A, title):
    xs = np.linspace(-3, 3, 400)
    ys = np.linspace(-3, 3, 400)
    X, Y = np.meshgrid(xs, ys)
    P = np.stack([X, Y], axis=-1)
    Z = 0.5 * np.einsum('...i,ij,...j->...', P, A, P)

    eigvals, eigvecs = np.linalg.eigh(A)

    plt.figure(figsize=(6, 5))
    # TODO: plt.contour(...)
    # TODO: нарисуйте два собственных вектора из начала координат через plt.arrow или plt.quiver
    # TODO: подпишите lambda_1, lambda_2
    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.gca().set_aspect('equal')
    plt.title(title)
    plt.show()

for theta in [0.0, np.pi/6, np.pi/3]:
    Q = rotation(theta)
    A = Q @ np.diag([1.0, 16.0]) @ Q.T
    # TODO: contour_for_A(A, ...)
    pass


---
## Задача 5. Число обусловленности и вытянутость линий уровня

Для $\kappa\in\{1,4,25,100\}$ постройте двумерную положительно определённую матрицу с собственными значениями $1$ и $\kappa$, затем случайно поверните её.

1. Для каждой матрицы вычислите
   $$
   \kappa_2(A)=\frac{\lambda_{\max}}{\lambda_{\min}}.
   $$
2. Постройте линию уровня
   $$
   \frac12x^\top Ax=1.
   $$
3. Найдите полуоси эллипса из собственных значений:
   $$
   a_i=\sqrt{\frac{2}{\lambda_i}}.
   $$
4. Проверьте
   $$
   \frac{a_{\max}}{a_{\min}}=\sqrt{\kappa_2(A)}.
   $$
5. Убедитесь экспериментально, что случайный поворот меняет ориентацию эллипса, но не его вытянутость.


In [ ]:
condition_numbers = [1, 4, 25, 100]

for kappa in condition_numbers:
    theta = rng.uniform(0, np.pi)
    Q = rotation(theta)
    A = Q @ np.diag([1.0, float(kappa)]) @ Q.T

    # TODO: eigvals = np.linalg.eigvalsh(A)
    # TODO: kappa_computed, semiaxes, axis_ratio

    xs = np.linspace(-2.2, 2.2, 400)
    ys = np.linspace(-2.2, 2.2, 400)
    X, Y = np.meshgrid(xs, ys)
    P = np.stack([X, Y], axis=-1)
    Z = 0.5 * np.einsum('...i,ij,...j->...', P, A, P)

    plt.figure(figsize=(5.5, 5))
    # TODO: одна линия уровня Z=1
    plt.axhline(0, linewidth=0.7)
    plt.axvline(0, linewidth=0.7)
    plt.gca().set_aspect('equal')
    plt.title(f'kappa = {kappa}')
    plt.show()


---
## Задача 6. Чувствительность решения $Ax=b$

Для каждого $\kappa\in\{1,10,100,1000\}$ постройте симметричную положительно определённую матрицу $A$ с таким числом обусловленности.

1. Выберите случайный ненулевой $b$ и найдите $x=A^{-1}b$.
2. Сгенерируйте 5000 малых случайных возмущений $\Delta b$ одинаковой относительной длины
   $$
   \frac{\|\Delta b\|_2}{\|b\|_2}=10^{-3}.
   $$
3. Для каждого возмущения найдите
   $$
   R=
   \frac{\|\Delta x\|_2/\|x\|_2}
        {\|\Delta b\|_2/\|b\|_2}.
   $$
4. Проверьте численно $R\le\kappa_2(A)$.
5. Отдельно постройте «наихудшую» конфигурацию: возьмите $b$ вдоль собственного вектора $v_{\max}$, а $\Delta b$ вдоль $v_{\min}$. Сравните полученный $R$ с $\kappa_2(A)$.
6. Для каждого $\kappa$ постройте отдельную гистограмму значений $R$.


In [ ]:
def spd_with_condition_number(d, kappa, rng):
    Q = random_orthogonal(d, rng)
    eigvals = np.geomspace(1.0, float(kappa), d)
    A = Q @ np.diag(eigvals) @ Q.T
    return A, eigvals, Q


d = 20
relative_noise = 1e-3
n_trials = 5000

for kappa in [1, 10, 100, 1000]:
    A, eigvals, Q = spd_with_condition_number(d, kappa, rng)
    b = rng.normal(size=d)
    x = np.linalg.solve(A, b)

    # TODO: сгенерировать n_trials направлений Delta_b и нормировать до нужной относительной длины
    # TODO: Delta_x можно получить решением A Delta_x = Delta_b
    # TODO: вычислить массив ratios R
    # TODO: проверить max(R) <= kappa с численным допуском

    # TODO: отдельная гистограмма R
    # plt.figure(...)
    # ...
    # plt.show()

    # TODO: worst-case: b || v_max, Delta_b || v_min


---
## Задача 7. Численная проверка происхождения градиентного шага

В пространстве $\mathbb R^{50}$ зафиксируйте случайный градиент $g$ и $\alpha>0$. Рассмотрите

$$
m(h)=g^\top h+\frac{1}{2\alpha}\|h\|_2^2.
$$

1. Реализуйте `m(h)` и её градиент по $h$.
2. Найдите минимум `m(h)` численно с помощью `scipy.optimize.minimize`.
3. Сравните численный минимум с точным решением
   $$
   h^\star=-\alpha g.
   $$
4. Повторите эксперимент для нескольких значений $\alpha$.
5. Проверьте, что минимальное значение модели равно
   $$
   m(h^\star)=-\frac{\alpha}{2}\|g\|_2^2.
   $$


In [ ]:
d = 50
g = rng.normal(size=d)


def solve_local_model(alpha):
    def m(h):
        return g @ h + 0.5 / alpha * (h @ h)

    def grad_m(h):
        # TODO
        return np.zeros_like(h)

    h0 = np.zeros(d)
    # TODO: res = minimize(m, h0, jac=grad_m, method='BFGS')
    # TODO: exact = -alpha * g
    # TODO: вернуть нормы ошибки и значения модели
    return None

for alpha in [0.01, 0.1, 0.5, 1.0]:
    # TODO: вызов solve_local_model(alpha)
    pass


---
## Задача 8. Градиентный спуск в собственном базисе

Сгенерируйте квадратичную задачу

$$
f(x)=\frac12x^\top Ax-b^\top x,
$$

где $A=Q\Lambda Q^\top\succ0$ и спектр $\Lambda$ известен заранее.

1. Найдите точное $x^\star$.
2. Запустите GD с шагом $\alpha=1/L$.
3. На каждой итерации вычисляйте ошибку $e_k=x_k-x^\star$ и её координаты в собственном базисе
   $$
   z_k=Q^\top e_k.
   $$
4. Для нескольких индексов $i$ сравните реальные $z_{k,i}$ с теоретическими
   $$
   z_{k,i}^{\rm theory}=(1-\alpha\lambda_i)^k z_{0,i}.
   $$
5. Постройте отдельный график для каждой выбранной компоненты.
6. Объясните, почему компоненты с разными $\lambda_i$ затухают с разной скоростью.


In [ ]:
d = 20
Q = random_orthogonal(d, rng)
eigvals = np.linspace(1.0, 20.0, d)
A = Q @ np.diag(eigvals) @ Q.T
b = rng.normal(size=d)
x_star = np.linalg.solve(A, b)

L = eigvals.max()
alpha = 1.0 / L
x0 = rng.normal(size=d)

n_steps = 80
# TODO: реализуйте GD и сохраните все x_k в список/массив trajectory
trajectory = []

# TODO: errors = trajectory - x_star
# TODO: Z = errors @ Q  (эквивалентно Q.T @ e для каждой строки)

selected = [0, 4, 9, 19]
for i in selected:
    # TODO: построить реальную и теоретическую компоненту на одном графике
    # theory = Z[0, i] * (1 - alpha * eigvals[i]) ** np.arange(...)
    pass


---
## Задача 9. Экспериментальная граница устойчивости шага

Возьмите положительно определённую квадратичную функцию с $L=5$. Теория утверждает, что GD сходится из любой начальной точки тогда и только тогда, когда

$$
0<\alpha<\frac{2}{L}=0.4.
$$

1. Переберите $\alpha$ на плотной сетке, например от $0.01$ до $0.6$.
2. Для каждого $\alpha$ выполните 100 итераций GD из одной и той же начальной точки.
3. Постройте зависимость
   $$
   \alpha\mapsto \|x_{100}-x^\star\|_2.
   $$
   Вертикальной линией отметьте $2/L$.
4. Отдельно исследуйте $\alpha=0.1,0.25,0.4,0.45$: постройте $\|e_k\|_2$ от $k$.
5. Для каждого из четырёх шагов вычислите коэффициенты $q_i=1-\alpha\lambda_i$ и сопоставьте их с наблюдаемым поведением.


In [ ]:
d = 8
Q = random_orthogonal(d, rng)
eigvals = np.linspace(1.0, 5.0, d)
A = Q @ np.diag(eigvals) @ Q.T
b = rng.normal(size=d)
x_star = np.linalg.solve(A, b)
x0 = rng.normal(size=d)
L = eigvals.max()


def run_gd(A, b, x0, alpha, n_steps):
    x = x0.copy()
    traj = [x.copy()]
    for _ in range(n_steps):
        # TODO: x = x - alpha * (A @ x - b)
        pass
    return np.array(traj)

alphas = np.linspace(0.01, 0.6, 180)
final_errors = []
for alpha in alphas:
    # TODO: traj = run_gd(...)
    # TODO: final_errors.append(...)
    pass

# TODO: отдельный график alpha -> final_error, вертикальная линия 2/L

for alpha in [0.1, 0.25, 0.4, 0.45]:
    # TODO: отдельный график ||e_k|| по k
    # TODO: вывести q_i и max |q_i|
    pass


---
## Задача 10. Обусловленность и скорость градиентного спуска

Сравним квадратичные задачи одинаковой размерности с

$$
\kappa\in\{2,10,50,100\}.
$$

Для каждой задачи используйте $\alpha=1/L$.

1. Постройте SPD-матрицу с $\lambda_{\min}=1$ и $\lambda_{\max}=\kappa$.
2. Запустите GD из случайной начальной точки и постройте
   $$
   \frac{\|e_k\|_2}{\|e_0\|_2}.
   $$
3. На тот же график добавьте теоретическую верхнюю оценку
   $$
   \left(1-\frac1\kappa\right)^k.
   $$
4. Повторите эксперимент для специально выбранной начальной ошибки $e_0=v_{\min}$. Что изменилось?
5. Для каждого $\kappa$ посчитайте фактическое число итераций до
   $$
   \|e_k\|_2\le10^{-2}\|e_0\|_2
   $$
   и сравните с оценкой из
   $$
   \left(1-\frac1\kappa\right)^k\le10^{-2}.
   $$
6. Сформулируйте итоговый вывод о связи числа обусловленности и скорости GD.


In [ ]:
d = 20
n_steps = 600
condition_numbers = [2, 10, 50, 100]

for kappa in condition_numbers:
    A, eigvals, Q = spd_with_condition_number(d, kappa, rng)
    b = rng.normal(size=d)
    x_star = np.linalg.solve(A, b)
    L = eigvals.max()
    alpha = 1.0 / L

    # Эксперимент A: случайная начальная точка
    x0 = rng.normal(size=d)
    # TODO: traj = run_gd(...)
    # TODO: relative_errors = ||x_k-x_star|| / ||x0-x_star||
    # TODO: bound = (1 - 1/kappa) ** np.arange(...)
    # TODO: отдельный semilogy-график relative_errors и bound

    # Эксперимент B: наихудшее направление e0 = v_min
    v_min = Q[:, 0]  # eigvals возрастают: первый столбец соответствует lambda_min=1
    x0_worst = x_star + v_min
    # TODO: повторите GD и сравните с bound

    # TODO: число итераций до 1e-2 и теоретическая оценка
    pass


---
## Итоговая самопроверка

После выполнения задач вы должны уметь:

- переходить между координатной и матричной записью квадратичной функции;
- классифицировать симметричную матрицу по её спектру;
- находить точный минимум положительно определённой квадратичной функции;
- читать геометрию линий уровня по собственным значениям и векторам;
- интерпретировать число обусловленности как меру вытянутости и чувствительности;
- выводить градиентный шаг из регуляризованной локальной модели;
- раскладывать динамику GD по собственным направлениям;
- проверять условие $0<\alpha<2/L$;
- объяснять, почему большой $\kappa$ замедляет градиентный спуск.
